<a href="https://colab.research.google.com/github/nyp-sit/dit-it3103/blob/main/week4/1.using_pretrained_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IT3103 Practical 04a: Using Pretrained CNN models

Welcome to this week's programming exercise. We have covered many different Convolutional Neural Network architectures such as VGG, ResNet, Inception and MobileNet. It is time to see them in action.

At the end of this exercise, you will be able to:
- load pretrained models of some popular Convolutional Neural Networks and use them to classify images
- identify some of the architecture patterns in the popular Convolutional Neural Network
- compare the inference speed of different models


## Get the sample image

We will use the pretrained model to classify a sample image (a picture of table and chair). Let's go ahead and download the image.

In [ ]:
# wget is a linux command available on linux os like Ubuntu
!wget https://nyp-aicourse.s3.ap-southeast-1.amazonaws.com/it3103/resources/chair_table.png

In [ ]:
## run on windows os, you can use the following code to download the image file

# import urllib.request
# urllib.request.urlretrieve('https://nyp-aicourse.s3.ap-southeast-1.amazonaws.com/it3103/resources/chair_table.png', 'chair_table.png')


In [ ]:
! pip install pandas

In [ ]:
from PIL import Image
import keras
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

%matplotlib inline

In [ ]:
# Read Images
img_path = 'chair_table.png'
img = keras.utils.load_img(img_path)

# display Images
plt.imshow(img)

## VGG16 - Pretrained Model

In [ ]:
from keras.applications import vgg16

vgg16_model = vgg16.VGG16(weights='imagenet')
vgg16_model.summary()

### *Exercise 1*
Look at the print out of model summary, answer the following questions: 
1. What is the expected input image size?
2. What are the last four layers in VGG-16?

<details><summary>Click here for answer</summary>
    
1. it is expected to have a height of 224 and width of 224
2. the last 4 layers are flatten (which flattens the 2-D array into 1-D array before feeding to FC layer), and 2 Fully-connected (Dense) layers, and the last layer is a soft-max layer to classify 1000-classes. This is quite typical of a image classifier.

</details>

Your answers:

In [ ]:
# Utility Function to Load Image, Preprocess input and Targets
def predict_image(model, img_path, preprocess_input_fn, decode_predictions_fn, target_size=(224,224)):
    img = keras.utils.load_img(img_path, target_size=target_size)
    x = keras.utils.img_to_array(img)
    x = np.expand_dims(x, axis=0)
    x = preprocess_input_fn(x)
    preds = model.predict(x)
    predictions_df = pd.DataFrame(decode_predictions_fn(preds, top=10)[0])
    predictions_df.columns = ["Predicted Class", "Name", "Probability"]
    return predictions_df

In [ ]:
#img_path="rocking_chair.png"  ## Uncomment this and put the path to your file here if desired
# Predict Results
predict_image(vgg16_model, img_path, vgg16.preprocess_input, vgg16.decode_predictions)

Notice that we pass in `vgg.preprocess_input` function to preprocess the image before calling `model.predict()`. Different network (e.g. VGG, ResNet, etc) expects the input image to be normalized in different ways, and different models will provide their own preprocess_input() function to perform the normalization.

We also call `np.expand_dims(x, axis=0)` before calling `preprocess_input()` and `predict()`.



### *Exercise 2*

1. What does `np.expand_dims(x, axis=0)` do and why do we need it?
2. Our sample picture consists of both table and chair. What does VGG16 predict? and why do you think it predicts so?
3. Of the top 10 predictions, did you see any prediction about chair?


<details><summary>Click here for answer</summary>

1. np.expand_dims() increases the number of dimensions and the axis of the new dimension is specified by the axis parameter. In this case, we add in a new axis as axis=0, first axis. This is because the preprocess_input() and predict() function expects the images to be in the shape (samples, height, width, channels), the 1st axis being the batch.

2. It predicts dining table. It probably focus on the object in the middle of the image.

3. Yes, folding_chair is one of the top 10 predictions.

</details>

Your answer:

## Resnet50 - Pretrained Model

In [ ]:
!pip install pydot graphviz

In [ ]:
# It will download the weights that might take a while
# Also, the summary will be quite long, since Resnet50 is a much larger network than VGG16

from keras.applications import ResNet50

resnet50_model = ResNet50(weights='imagenet')

# let's plot the model, instead of using model.summary(), as it is easier to see some of the skip connections
keras.utils.plot_model(resnet50_model, to_file="resnet.png")

In [ ]:
resnet50_model.summary()

### *Exercise 3*

1. Can you identify the skip connection block from the model plot()?
2. Look at the last few layers in the ResNet. How are they different from those of VGG-16?

<details><summary>Click here for answer</summary>
    
1. Look for those 'Add' layer (e.g. layer with name add_2). The Add layer adds the skip connection with the previous layer. Notice that the add is done before the Activation function.

2. ResNet does not use make use of Full-connected layers as classification layers. Instead it replaces the FC layers with GlobalAveragePooling2D. This architecture is very common in more modern architectures.

</details>

Your answers:

In [ ]:
# Predict Results
from keras.applications import resnet
predict_image(resnet50_model, img_path, resnet.preprocess_input, resnet.decode_predictions)

## MobileNet v1 - Pretrained Model

In [ ]:
from keras.applications import mobilenet
mobilenet_model = mobilenet.MobileNet(weights='imagenet')

# plot the model
keras.utils.plot_model(mobilenet_model, to_file="mobilenet.png")

In [ ]:
mobilenet_model.summary()

### *Exercise 4*

1. Can you identify the **Depth-wise** seperable Convolution layer from the model summary()?
2. How about the **Point-wise** convolution?
3. Look at the last few layers in the MobileNet. How are they different from those of VGG-16?

<details><summary>Click here for answer</summary>

1. For example, the layer called 'conv_dw1'.

2. For example, the layer called 'conv_pw1'.

3. MobileNet does not use make use of Full-connected layers as classification layers. Instead it replaces the FC layers with GlobalAveragePooling2D. This architecture is very common in more modern architectures.
    
</details>

Your answers:

In [ ]:
predict_image(mobilenet_model, img_path, mobilenet.preprocess_input, mobilenet.decode_predictions)


## Speed comparison

We compare the inference speed of the three different models. Which one has the fastest inference speed?

In [ ]:
%timeit predict_image(vgg16_model, img_path, vgg16.preprocess_input, vgg16.decode_predictions)


In [ ]:
%timeit predict_image(resnet50_model, img_path, resnet.preprocess_input, resnet.decode_predictions)


In [ ]:
%timeit predict_image(mobilenet_model, img_path, mobilenet.preprocess_input, mobilenet.decode_predictions)

## Additional Exercise 1 (Optional): InceptionV3

Load the InceptionV3 pretrained model and use it to classify the same sample image.

- Load InceptionV3 using `keras.applications.InceptionV3(weights='imagenet')`
- Use `keras.applications.inception_v3.preprocess_input` and `keras.applications.inception_v3.decode_predictions`
- Note: InceptionV3 expects input size of **(299, 299)**, not (224, 224)
- Compare the top predictions with VGG16, ResNet50, and MobileNet
- Use `%timeit` to compare inference speed

<details>
<summary>Click here for answer</summary>

```python
from keras.applications import inception_v3

inception_model = inception_v3.InceptionV3(weights='imagenet')

# Note the different target_size for InceptionV3
predict_image(inception_model, img_path,
              inception_v3.preprocess_input,
              inception_v3.decode_predictions,
              target_size=(299, 299))
```

</details>

In [ ]:
## TODO: Load InceptionV3 and classify the sample image
## Remember to use target_size=(299, 299)



In [ ]:
## TODO: Compare inference speed of InceptionV3 with the other models



## Additional Exercise 2 (Optional): Identify Architectural Patterns

Examine the InceptionV3 model summary and answer the following:

1. Can you identify the **Inception modules** from the model summary? What layer names indicate the parallel branches being concatenated?
2. Does InceptionV3 use Fully Connected layers or Global Average Pooling at the end? How does this compare with VGG16?
3. Does InceptionV3 use **1x1 convolutions**? Find an example layer name.

<details>
<summary>Click here for answer</summary>

1. Look for layers with names like `mixed0`, `mixed1`, etc. These are the Inception modules. Inside each module, you can see `concatenate` layers that merge the parallel branches (1x1, 3x3, 5x5 convolutions and pooling).

2. InceptionV3 uses GlobalAveragePooling2D, not Fully Connected layers (unlike VGG16 which has two Dense layers with 4096 units each). This significantly reduces the number of parameters.

3. Yes, many layers use 1x1 convolutions (kernel size of 1). For example, layers with names like `conv2d_xx` with output depth different from input depth but same spatial dimensions. These are used to reduce the depth before applying larger convolutions (as covered in lecture slides 9-10).

</details>

In [ ]:
## TODO: Print InceptionV3 model summary and identify the architectural patterns

